# Music Play Enrichment with Mistral 7B

This notebook generates semantic enrichments (mood, instruments, genres, style tags) for KEXP music plays using a local LLM on Colab.

**Model:** Mistral 7B Instruct v0.3 (4-bit quantized)
- Fits on free T4 GPU (~6GB VRAM)
- Fast inference with vLLM batching
- Guaranteed JSON output with Outlines

**Output:** Structured enrichment data:
- Mood tags (energetic, melancholic, uplifting, etc.)
- Instrument detection (guitar, synth, drums, etc.)
- Genre classification (indie rock, electronic, jazz, etc.)
- Style descriptors (lo-fi, atmospheric, driving, etc.)

**Requirements:**
1. Enable GPU runtime (Runtime -> Change runtime type -> T4 GPU)
2. Upload `enriched_plays_full.csv` to Google Drive (or use existing)
3. Run all cells

**Processing:** ~2.2M plays at ~500 plays/min = ~73 hours total
- Checkpoints every 10,000 plays
- Resume from last checkpoint if interrupted

In [ ]:
# Install dependencies
# Note: vLLM requires specific CUDA versions - Colab T4 has CUDA 12.x
!pip install -q vllm outlines pydantic pandas tqdm

# Alternative if vLLM has issues: use transformers with bitsandbytes
# !pip install -q transformers accelerate bitsandbytes outlines pydantic pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import os

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA version: {torch.version.cuda}")

# Configure paths
DRIVE_DIR = '/content/drive/MyDrive/music'
os.makedirs(DRIVE_DIR, exist_ok=True)

## Define Enrichment Schema

Pydantic schema ensures structured output. Outlines uses this to constrain LLM generation.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

# Predefined tag vocabularies for consistent output
class MoodTag(str, Enum):
    ENERGETIC = "energetic"
    MELANCHOLIC = "melancholic"
    UPLIFTING = "uplifting"
    AGGRESSIVE = "aggressive"
    CALM = "calm"
    DREAMY = "dreamy"
    DARK = "dark"
    PLAYFUL = "playful"
    NOSTALGIC = "nostalgic"
    INTENSE = "intense"
    GROOVY = "groovy"
    ETHEREAL = "ethereal"
    RAW = "raw"
    WARM = "warm"
    COLD = "cold"

class InstrumentTag(str, Enum):
    GUITAR = "guitar"
    BASS = "bass"
    DRUMS = "drums"
    SYNTH = "synth"
    PIANO = "piano"
    VOCALS = "vocals"
    STRINGS = "strings"
    HORNS = "horns"
    ELECTRONIC = "electronic"
    ACOUSTIC = "acoustic"
    SAXOPHONE = "saxophone"
    TRUMPET = "trumpet"
    VIOLIN = "violin"
    ORGAN = "organ"
    PERCUSSION = "percussion"

class GenreTag(str, Enum):
    INDIE_ROCK = "indie rock"
    ELECTRONIC = "electronic"
    HIP_HOP = "hip hop"
    JAZZ = "jazz"
    FOLK = "folk"
    PUNK = "punk"
    METAL = "metal"
    RNB = "r&b"
    POP = "pop"
    AMBIENT = "ambient"
    SOUL = "soul"
    WORLD = "world"
    CLASSICAL = "classical"
    BLUES = "blues"
    REGGAE = "reggae"
    COUNTRY = "country"
    EXPERIMENTAL = "experimental"
    SHOEGAZE = "shoegaze"
    POST_PUNK = "post-punk"
    PSYCHEDELIC = "psychedelic"

class StyleTag(str, Enum):
    LO_FI = "lo-fi"
    ATMOSPHERIC = "atmospheric"
    DRIVING = "driving"
    MINIMALIST = "minimalist"
    LAYERED = "layered"
    DISTORTED = "distorted"
    CLEAN = "clean"
    VINTAGE = "vintage"
    MODERN = "modern"
    DANCEABLE = "danceable"
    MELODIC = "melodic"
    RHYTHMIC = "rhythmic"
    SPARSE = "sparse"
    LUSH = "lush"
    GRITTY = "gritty"


class MusicEnrichment(BaseModel):
    """Structured enrichment for a music play."""
    
    moods: List[MoodTag] = Field(
        ...,
        min_length=1,
        max_length=3,
        description="Primary mood tags (1-3)"
    )
    instruments: List[InstrumentTag] = Field(
        ...,
        min_length=1,
        max_length=5,
        description="Detected instruments (1-5)"
    )
    genres: List[GenreTag] = Field(
        ...,
        min_length=1,
        max_length=3,
        description="Genre classifications (1-3)"
    )
    styles: List[StyleTag] = Field(
        ...,
        min_length=1,
        max_length=3,
        description="Style descriptors (1-3)"
    )
    energy_level: int = Field(
        ...,
        ge=1,
        le=10,
        description="Energy level 1-10 (1=very calm, 10=very energetic)"
    )
    danceability: int = Field(
        ...,
        ge=1,
        le=10,
        description="Danceability 1-10 (1=not danceable, 10=very danceable)"
    )

print("Schema defined. Tags available:")
print(f"  Moods: {len(MoodTag)} options")
print(f"  Instruments: {len(InstrumentTag)} options")
print(f"  Genres: {len(GenreTag)} options")
print(f"  Styles: {len(StyleTag)} options")

## Load Model with vLLM

vLLM provides 24x faster batched inference through continuous batching and PagedAttention.

In [ ]:
# Option 1: vLLM (fastest, but may have CUDA compatibility issues)
try:
    from vllm import LLM, SamplingParams
    import outlines
    from outlines import models, generate
    
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
    
    print(f"Loading {MODEL_NAME} with vLLM...")
    
    # Load with 4-bit quantization for T4 GPU
    model = outlines.models.vllm(
        MODEL_NAME,
        quantization="awq",  # 4-bit quantization
        dtype="half",
        max_model_len=2048,
        gpu_memory_utilization=0.9
    )
    
    USE_VLLM = True
    print("vLLM loaded successfully!")
    
except Exception as e:
    print(f"vLLM failed: {e}")
    print("Falling back to transformers...")
    USE_VLLM = False

In [ ]:
# Option 2: Transformers fallback (slower but more compatible)
if not USE_VLLM:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import outlines
    from outlines import models, generate
    
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
    
    print(f"Loading {MODEL_NAME} with transformers + bitsandbytes...")
    
    # 4-bit quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    
    # Load model
    hf_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # Wrap for Outlines
    model = outlines.models.Transformers(hf_model, tokenizer)
    
    print("Transformers model loaded successfully!")

In [ ]:
# Create structured generator with Outlines
# This guarantees valid JSON output matching our Pydantic schema

generator = generate.json(model, MusicEnrichment)

print("Structured generator ready!")
print("Output will always be valid MusicEnrichment JSON.")

## Test the Generator

In [ ]:
def create_prompt(artist: str, song: str, album: str = None, comment: str = None) -> str:
    """Create enrichment prompt for a music play."""
    
    parts = [f"Artist: {artist}", f"Song: {song}"]
    if album:
        parts.append(f"Album: {album}")
    if comment:
        parts.append(f"DJ Comment: {comment}")
    
    track_info = "\n".join(parts)
    
    prompt = f"""[INST] You are a music expert. Analyze this track and provide enrichment tags.

{track_info}

Based on the artist, song title, album, and any DJ comments, classify this track with:
- moods: 1-3 mood tags from [energetic, melancholic, uplifting, aggressive, calm, dreamy, dark, playful, nostalgic, intense, groovy, ethereal, raw, warm, cold]
- instruments: 1-5 likely instruments from [guitar, bass, drums, synth, piano, vocals, strings, horns, electronic, acoustic, saxophone, trumpet, violin, organ, percussion]
- genres: 1-3 genre tags from [indie rock, electronic, hip hop, jazz, folk, punk, metal, r&b, pop, ambient, soul, world, classical, blues, reggae, country, experimental, shoegaze, post-punk, psychedelic]
- styles: 1-3 style descriptors from [lo-fi, atmospheric, driving, minimalist, layered, distorted, clean, vintage, modern, danceable, melodic, rhythmic, sparse, lush, gritty]
- energy_level: 1-10 (1=very calm, 10=very energetic)
- danceability: 1-10 (1=not danceable, 10=very danceable)

Respond with valid JSON only. [/INST]"""
    
    return prompt

# Test with a sample
test_prompt = create_prompt(
    artist="Radiohead",
    song="Everything In Its Right Place",
    album="Kid A",
    comment="Haunting opener from their electronic masterpiece"
)

print("Test prompt:")
print(test_prompt[:500] + "...")

In [ ]:
# Generate test enrichment
print("Generating test enrichment...")

test_result = generator(test_prompt)

print("\nResult:")
print(f"  Moods: {[m.value for m in test_result.moods]}")
print(f"  Instruments: {[i.value for i in test_result.instruments]}")
print(f"  Genres: {[g.value for g in test_result.genres]}")
print(f"  Styles: {[s.value for s in test_result.styles]}")
print(f"  Energy: {test_result.energy_level}/10")
print(f"  Danceability: {test_result.danceability}/10")

## Load Play Data

In [ ]:
import pandas as pd

# Load plays data
INPUT_FILE = f'{DRIVE_DIR}/enriched_plays_full.csv'

print(f"Loading plays from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE, low_memory=False)

print(f"Loaded {len(df):,} plays")
print(f"\nColumns: {list(df.columns)}")

# Show sample
print(f"\nSample play:")
sample = df.iloc[0]
print(f"  Artist: {sample['artist']}")
print(f"  Song: {sample['song']}")
print(f"  Album: {sample.get('album', 'N/A')}")
print(f"  Comment: {sample.get('comment', 'N/A')[:100] if pd.notna(sample.get('comment')) else 'N/A'}")

## Batch Processing with Checkpointing

In [ ]:
import json
import time
from tqdm import tqdm
from pathlib import Path

# Configuration
BATCH_SIZE = 8 if USE_VLLM else 4  # Smaller batches for transformers
CHECKPOINT_INTERVAL = 10000  # Save every 10k plays
OUTPUT_FILE = f'{DRIVE_DIR}/play_enrichments.jsonl'
CHECKPOINT_FILE = f'{DRIVE_DIR}/enrichment_checkpoint.json'

def load_checkpoint():
    """Load processing checkpoint."""
    if Path(CHECKPOINT_FILE).exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'last_processed_idx': -1, 'total_processed': 0}

def save_checkpoint(idx: int, total: int):
    """Save processing checkpoint."""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'last_processed_idx': idx, 'total_processed': total}, f)

def process_batch(plays: list) -> list:
    """Process a batch of plays and return enrichments."""
    prompts = []
    for play in plays:
        prompt = create_prompt(
            artist=play['artist'],
            song=play['song'],
            album=play.get('album') if pd.notna(play.get('album')) else None,
            comment=play.get('comment') if pd.notna(play.get('comment')) else None
        )
        prompts.append(prompt)
    
    # Generate all at once (batched)
    try:
        if USE_VLLM:
            # vLLM handles batching internally
            results = [generator(p) for p in prompts]
        else:
            # Transformers - process one at a time
            results = [generator(p) for p in prompts]
        
        enrichments = []
        for play, result in zip(plays, results):
            enrichments.append({
                'play_id': int(play['id']),
                'moods': [m.value for m in result.moods],
                'instruments': [i.value for i in result.instruments],
                'genres': [g.value for g in result.genres],
                'styles': [s.value for s in result.styles],
                'energy_level': result.energy_level,
                'danceability': result.danceability
            })
        return enrichments
    
    except Exception as e:
        print(f"Batch error: {e}")
        # Return empty results for failed batch
        return [{'play_id': int(p['id']), 'error': str(e)} for p in plays]

print(f"Batch size: {BATCH_SIZE}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL:,}")
print(f"Output file: {OUTPUT_FILE}")

In [ ]:
# Main processing loop
checkpoint = load_checkpoint()
start_idx = checkpoint['last_processed_idx'] + 1
total_processed = checkpoint['total_processed']

print(f"Starting from index {start_idx:,}")
print(f"Previously processed: {total_processed:,}")
print(f"Remaining: {len(df) - start_idx:,}")

# Open output file in append mode
start_time = time.time()
batch_times = []

with open(OUTPUT_FILE, 'a') as f:
    # Process in batches
    for batch_start in tqdm(range(start_idx, len(df), BATCH_SIZE), 
                           desc="Processing", 
                           total=(len(df) - start_idx) // BATCH_SIZE):
        
        batch_end = min(batch_start + BATCH_SIZE, len(df))
        batch = df.iloc[batch_start:batch_end].to_dict('records')
        
        batch_start_time = time.time()
        enrichments = process_batch(batch)
        batch_time = time.time() - batch_start_time
        batch_times.append(batch_time)
        
        # Write results
        for enrichment in enrichments:
            f.write(json.dumps(enrichment) + '\n')
        
        total_processed += len(enrichments)
        
        # Checkpoint
        if total_processed % CHECKPOINT_INTERVAL < BATCH_SIZE:
            save_checkpoint(batch_end - 1, total_processed)
            f.flush()
            
            # Progress stats
            elapsed = time.time() - start_time
            rate = total_processed / elapsed * 60  # plays per minute
            remaining = len(df) - batch_end
            eta_minutes = remaining / rate if rate > 0 else 0
            
            print(f"\n[Checkpoint] Processed: {total_processed:,} | "
                  f"Rate: {rate:.0f}/min | "
                  f"ETA: {eta_minutes/60:.1f}h")

# Final checkpoint
save_checkpoint(len(df) - 1, total_processed)

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"COMPLETE!")
print(f"{'='*60}")
print(f"Total processed: {total_processed:,}")
print(f"Time: {elapsed/3600:.1f} hours")
print(f"Rate: {total_processed/elapsed*60:.0f} plays/min")
print(f"Output: {OUTPUT_FILE}")

## Convert to Upload Format

Convert JSONL output to format expected by FAISS API enrichment endpoint.

In [ ]:
import json

# Read all enrichments
enrichments = []
with open(OUTPUT_FILE, 'r') as f:
    for line in f:
        enrichments.append(json.loads(line))

print(f"Loaded {len(enrichments):,} enrichments")

# Check for errors
errors = [e for e in enrichments if 'error' in e]
success = [e for e in enrichments if 'error' not in e]

print(f"Successful: {len(success):,}")
print(f"Errors: {len(errors):,}")

if errors:
    print(f"\nSample error: {errors[0]}")

In [ ]:
# Convert to API format and split into chunks for upload
CHUNK_SIZE = 10000  # Upload in chunks of 10k

def convert_to_api_format(enrichments: list) -> list:
    """Convert enrichments to API import format."""
    return [
        {
            'play_id': e['play_id'],
            'data': {
                'moods': e['moods'],
                'instruments': e['instruments'],
                'genres': e['genres'],
                'styles': e['styles'],
                'energy_level': e['energy_level'],
                'danceability': e['danceability']
            }
        }
        for e in enrichments
        if 'error' not in e
    ]

api_enrichments = convert_to_api_format(success)
print(f"Converted {len(api_enrichments):,} enrichments to API format")

# Split into chunks
chunks = [
    api_enrichments[i:i+CHUNK_SIZE] 
    for i in range(0, len(api_enrichments), CHUNK_SIZE)
]

print(f"Split into {len(chunks)} chunks of up to {CHUNK_SIZE:,} each")

# Save chunks
CHUNKS_DIR = f'{DRIVE_DIR}/enrichment_chunks'
os.makedirs(CHUNKS_DIR, exist_ok=True)

for i, chunk in enumerate(chunks):
    chunk_file = f'{CHUNKS_DIR}/chunk_{i:04d}.json'
    with open(chunk_file, 'w') as f:
        json.dump({
            'enrichment_type': 'llm_tags',
            'enrichments': chunk
        }, f)
    print(f"Saved {chunk_file} ({len(chunk):,} items)")

print(f"\nChunks saved to {CHUNKS_DIR}/")
print("Upload each chunk to: POST /api/enrichments")

## Upload to FAISS API (Optional)

If your FAISS API is accessible, upload enrichments directly.

In [ ]:
import httpx
import asyncio

# FAISS API URL (update this)
FAISS_API_URL = "https://your-faiss-api.com"  # TODO: Update this

async def upload_chunk(chunk_file: str, client: httpx.AsyncClient):
    """Upload a single chunk to the API."""
    with open(chunk_file, 'r') as f:
        data = json.load(f)
    
    response = await client.post(
        f"{FAISS_API_URL}/api/enrichments",
        json=data,
        timeout=300  # 5 min timeout for large chunks
    )
    
    if response.status_code == 200:
        result = response.json()
        return True, result.get('count', 0)
    else:
        return False, response.text

async def upload_all_chunks():
    """Upload all chunks to the API."""
    chunk_files = sorted(Path(CHUNKS_DIR).glob('chunk_*.json'))
    
    print(f"Uploading {len(chunk_files)} chunks to {FAISS_API_URL}...")
    
    async with httpx.AsyncClient() as client:
        total_uploaded = 0
        
        for chunk_file in tqdm(chunk_files, desc="Uploading"):
            success, result = await upload_chunk(str(chunk_file), client)
            
            if success:
                total_uploaded += result
            else:
                print(f"Failed: {chunk_file} - {result}")
        
        print(f"\nTotal uploaded: {total_uploaded:,}")

# Uncomment to run upload
# await upload_all_chunks()

## Statistics and Sample Output

In [ ]:
from collections import Counter

# Analyze enrichment distribution
all_moods = []
all_instruments = []
all_genres = []
all_styles = []
energies = []
danceabilities = []

for e in success:
    all_moods.extend(e.get('moods', []))
    all_instruments.extend(e.get('instruments', []))
    all_genres.extend(e.get('genres', []))
    all_styles.extend(e.get('styles', []))
    if 'energy_level' in e:
        energies.append(e['energy_level'])
    if 'danceability' in e:
        danceabilities.append(e['danceability'])

print("=" * 60)
print("ENRICHMENT STATISTICS")
print("=" * 60)

print(f"\nTop 10 Moods:")
for mood, count in Counter(all_moods).most_common(10):
    print(f"  {mood}: {count:,} ({count/len(success)*100:.1f}%)")

print(f"\nTop 10 Instruments:")
for inst, count in Counter(all_instruments).most_common(10):
    print(f"  {inst}: {count:,} ({count/len(success)*100:.1f}%)")

print(f"\nTop 10 Genres:")
for genre, count in Counter(all_genres).most_common(10):
    print(f"  {genre}: {count:,} ({count/len(success)*100:.1f}%)")

print(f"\nTop 10 Styles:")
for style, count in Counter(all_styles).most_common(10):
    print(f"  {style}: {count:,} ({count/len(success)*100:.1f}%)")

if energies:
    import numpy as np
    print(f"\nEnergy Level: mean={np.mean(energies):.1f}, median={np.median(energies):.0f}")
    print(f"Danceability: mean={np.mean(danceabilities):.1f}, median={np.median(danceabilities):.0f}")

In [ ]:
# Show sample enrichments
import random

print("Sample Enrichments:")
print("=" * 60)

for e in random.sample(success, min(5, len(success))):
    play = df[df['id'] == e['play_id']].iloc[0]
    print(f"\n{play['artist']} - {play['song']}")
    print(f"  Moods: {e['moods']}")
    print(f"  Instruments: {e['instruments']}")
    print(f"  Genres: {e['genres']}")
    print(f"  Styles: {e['styles']}")
    print(f"  Energy: {e['energy_level']}/10, Dance: {e['danceability']}/10")